# W02 – SQL esencial I en DuckDB (SELECT/WHERE/GROUP BY/NULLs)

## Conexión con DDIA
- **DDIA Cap. 2**: modelos de datos y lenguajes de consulta (SQL como herramienta central).
- Aquí convertimos *preguntas* en *consultas* sobre un dataset real (NASA Exoplanet Archive).

## Prerrequisitos
- Haber hecho W01A y W01B (o al menos tener Python + dependencias instaladas).
- Si no tienes `data/raw/pscomppars.csv`, el notebook lo descargará.

## Objetivos
- Crear una **vista** `raw_ps` desde un CSV.
- Usar SQL básico: `SELECT`, `WHERE`, `ORDER BY`, `LIMIT`, `GROUP BY`, `HAVING`.
- Entender `NULL`: `COUNT(*)` vs `COUNT(col)` y `COALESCE`.

## Checklist de evidencias
- [ ] Output de: `SELECT count(*) FROM raw_ps`
- [ ] 6 consultas resueltas en la sección **TU TURNO**
- [ ] 10 consultas adicionales (tarea) guardadas al final


In [1]:
import os
os.chdir("..")
os.getcwd()

'c:\\Users\\camil\\Desktop\\semestre 11\\ingenieria de datos pc'

In [3]:
# Setup común (cross-platform)
import sys, subprocess
from pathlib import Path
import duckdb

DB_PATH = Path("data/exoplanets.duckdb")
DB_PATH.parent.mkdir(parents=True, exist_ok=True)
con = duckdb.connect(str(DB_PATH))

def run_module(mod: str, *args: str):
    cmd = [sys.executable, "-m", mod, *args]
    print("Running:", " ".join(cmd))
    subprocess.check_call(cmd)

raw_csv = Path("data/raw/pscomppars.csv")
if not raw_csv.exists():
    # Para clase: descarga razonable. Quita --limit si quieres el subset completo.
    run_module("src.ingest.download_exoplanets", "--format", "csv", "--limit", "50000")

# DuckDB no permite parámetros preparados en DDL (ej. CREATE VIEW).
# Insertamos la ruta como literal SQL, escapando comillas simples.
def sql_quote(s: str) -> str:
    return "'" + s.replace("'", "''") + "'"

raw_csv_abs = raw_csv.resolve()
con.execute(
    f"CREATE OR REPLACE VIEW raw_ps AS SELECT * FROM read_csv_auto({sql_quote(raw_csv_abs.as_posix())})"
)
con.execute("SELECT count(*) AS n_rows FROM raw_ps").fetchall()


[(6107,)]

## DEMO

In [4]:
# DEMO 1: inspección rápida
con.sql("DESCRIBE raw_ps").show()


┌─────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│   column_name   │ column_type │  null   │   key   │ default │  extra  │
│     varchar     │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ pl_name         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ hostname        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ discoverymethod │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ disc_year       │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ sy_snum         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ sy_pnum         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ sy_dist         │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ ra              │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ dec             │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ pl_orbper       │ DOUBLE      │ YES 

In [5]:
# DEMO 2: SELECT + LIMIT (muestra pequeña)
con.sql("""
SELECT pl_name, hostname, discoverymethod, disc_year
FROM raw_ps
LIMIT 10
""").show()


┌───────────────┬─────────────┬─────────────────┬───────────┐
│    pl_name    │  hostname   │ discoverymethod │ disc_year │
│    varchar    │   varchar   │     varchar     │   int64   │
├───────────────┼─────────────┼─────────────────┼───────────┤
│ Kepler-1167 b │ Kepler-1167 │ Transit         │      2016 │
│ Kepler-1740 b │ Kepler-1740 │ Transit         │      2021 │
│ Kepler-1581 b │ Kepler-1581 │ Transit         │      2016 │
│ Kepler-644 b  │ Kepler-644  │ Transit         │      2016 │
│ Kepler-1752 b │ Kepler-1752 │ Transit         │      2021 │
│ Kepler-280 c  │ Kepler-280  │ Transit         │      2014 │
│ Kepler-1208 b │ Kepler-1208 │ Transit         │      2016 │
│ Kepler-263 c  │ Kepler-263  │ Transit         │      2014 │
│ Kepler-1101 b │ Kepler-1101 │ Transit         │      2016 │
│ HD 168746 b   │ HD 168746   │ Radial Velocity │      2002 │
├───────────────┴─────────────┴─────────────────┴───────────┤
│ 10 rows                                         4 columns │
└───────

In [6]:
# DEMO 3: WHERE + ORDER BY (evita NULL)
con.sql("""
SELECT pl_name, pl_orbper, pl_rade
FROM raw_ps
WHERE pl_orbper IS NOT NULL
ORDER BY pl_orbper ASC
LIMIT 10
""").show()


┌──────────────────┬─────────────┬─────────────┐
│     pl_name      │  pl_orbper  │   pl_rade   │
│     varchar      │   double    │   double    │
├──────────────────┼─────────────┼─────────────┤
│ PSR J1719-1438 b │ 0.090706293 │        NULL │
│ ZTF J1828+2308 b │   0.1120067 │ 11.13051784 │
│ M62H b           │ 0.132935028 │        NULL │
│ KOI-1843.03      │   0.1768913 │        0.61 │
│ K2-137 b         │    0.179719 │        0.64 │
│ KIC 10001893 b   │      0.2197 │        NULL │
│ ZTF J1230-2655 b │  0.23597766 │ 13.78704626 │
│ TOI-6255 b       │  0.23818244 │       1.079 │
│ KOI-55 b         │    0.240104 │       0.759 │
│ TOI-6324 b       │    0.279221 │       1.059 │
├──────────────────┴─────────────┴─────────────┤
│ 10 rows                            3 columns │
└──────────────────────────────────────────────┘



In [10]:
# DEMO 4: NULLs — COUNT(*) vs COUNT(col)
con.sql("""
SELECT
  COUNT(*)                    AS total_rows,
  COUNT(pl_rade)              AS non_null_radius,
  COUNT(*) - COUNT(pl_rade)   AS null_radius
FROM raw_ps
""").show()


┌────────────┬─────────────────┬─────────────┐
│ total_rows │ non_null_radius │ null_radius │
│   int64    │      int64      │    int64    │
├────────────┼─────────────────┼─────────────┤
│       6107 │            6057 │          50 │
└────────────┴─────────────────┴─────────────┘



In [8]:
# DEMO 5: GROUP BY + agregados
con.sql("""
SELECT
  discoverymethod,
  COUNT(*) AS n_planets,
  AVG(pl_rade) AS avg_radius_earth
FROM raw_ps
GROUP BY 1
ORDER BY n_planets DESC
LIMIT 10
""").show()


┌───────────────────────────────┬───────────┬────────────────────┐
│        discoverymethod        │ n_planets │  avg_radius_earth  │
│            varchar            │   int64   │       double       │
├───────────────────────────────┼───────────┼────────────────────┤
│ Transit                       │      4501 │  4.368151792099985 │
│ Radial Velocity               │      1166 │  9.759872661948629 │
│ Microlensing                  │       266 │  9.850563909774435 │
│ Imaging                       │        92 │ 15.612215793749998 │
│ Transit Timing Variations     │        39 │  6.493408364210527 │
│ Eclipse Timing Variations     │        17 │ 12.893333333333334 │
│ Orbital Brightness Modulation │         9 │            9.64504 │
│ Pulsar Timing                 │         8 │  5.411333333333332 │
│ Astrometry                    │         6 │ 12.450000000000001 │
│ Pulsation Timing Variations   │         2 │              12.75 │
├───────────────────────────────┴───────────┴─────────────────

In [9]:
# DEMO 6: HAVING (filtra grupos después de agrupar)
con.sql("""
SELECT
  disc_year,
  COUNT(*) AS n
FROM raw_ps
WHERE disc_year IS NOT NULL
GROUP BY 1
HAVING COUNT(*) >= 200
ORDER BY disc_year ASC
""").show()


┌───────────┬───────┐
│ disc_year │   n   │
│   int64   │ int64 │
├───────────┼───────┤
│      2014 │   869 │
│      2016 │  1496 │
│      2018 │   315 │
│      2020 │   234 │
│      2021 │   564 │
│      2022 │   369 │
│      2023 │   324 │
│      2024 │   259 │
│      2025 │   240 │
└───────────┴───────┘



### Resumen
- `GROUP BY` cambia el grano: ya no son planetas, son **grupos**.
- `COUNT(*)` cuenta filas; `COUNT(col)` ignora `NULL`.
- `HAVING` filtra **después** de agrupar (a diferencia de `WHERE`).

---

## TU TURNO (práctica guiada)
Resuelve estas consultas. Pega el output (al menos las primeras filas) en cada celda.


### 1) ¿Cuántos planetas hay por año? (top 15 años con más planetas)

In [11]:
# TODO (1): ¿Cuántos planetas hay por año? (top 15 años con más planetas)
# Pistas: usa disc_year, filtra IS NOT NULL, GROUP BY, ORDER BY n DESC, LIMIT 15

query = """
SELECT
  disc_year,
  COUNT(*) AS n
FROM raw_ps
WHERE disc_year IS NOT NULL
GROUP BY disc_year
ORDER BY n DESC
LIMIT 15
"""
con.execute(query).fetchall()

[(2016, 1496),
 (2014, 869),
 (2021, 564),
 (2022, 369),
 (2023, 324),
 (2018, 315),
 (2024, 259),
 (2025, 240),
 (2020, 234),
 (2019, 196),
 (2015, 155),
 (2017, 152),
 (2012, 139),
 (2011, 135),
 (2013, 128)]

### 2) Top 10 sistemas (hostname) con más planetas

In [13]:
# TODO (2): Top 10 sistemas (hostname) con más planetas
# Pistas: GROUP BY hostname, cuenta filas, ORDER BY DESC, LIMIT 10

query = """
SELECT
  hostname,
  COUNT(*) AS n
FROM raw_ps
WHERE hostname IS NOT NULL
GROUP BY hostname
ORDER BY n DESC
LIMIT 10
"""
con.execute(query).fetchall()

[('KOI-351', 8),
 ('TRAPPIST-1', 7),
 ('TOI-178', 6),
 ('Kepler-80', 6),
 ('Kepler-20', 6),
 ('HD 10180', 6),
 ('HD 34445', 6),
 ('HD 110067', 6),
 ('Kepler-11', 6),
 ('HIP 41378', 6)]

### 3) ¿Qué fracción de filas tiene `pl_bmasse` nulo?

In [14]:
# TODO (3): ¿Qué fracción de filas tiene pl_bmasse nulo?
# Pistas: COUNT(*) total, COUNT(pl_bmasse) non_null, nulls = total - non_null
#       fracción = nulls / total (convierte a DOUBLE)

query = """
SELECT
  COUNT(*) AS total,
  COUNT(pl_bmasse) AS non_null,
  COUNT(*) - COUNT(pl_bmasse) AS nulls,
  (COUNT(*) - COUNT(pl_bmasse))::DOUBLE / COUNT(*) AS fraccion
FROM raw_ps
"""
con.execute(query).fetchall()

[(6107, 6076, 31, 0.005076142131979695)]

### 4) 10 planetas con mayor radio (pl_rade) (evita NULL)

In [15]:
# TODO (4): 10 planetas con mayor radio (pl_rade) (evita NULL)
# Pistas: WHERE pl_rade IS NOT NULL, ORDER BY pl_rade DESC, LIMIT 10

query = """
SELECT
  pl_name,
  pl_rade
FROM raw_ps
WHERE pl_rade IS NOT NULL
ORDER BY pl_rade DESC
LIMIT 10
"""
con.execute(query).fetchall()

[('V2376 Ori b', 87.20586985),
 ('HD 100546 b', 77.3421),
 ('GQ Lup b', 33.6),
 ('Kepler-297 d', 32.6),
 ('PDS 70 b', 30.48848),
 ('DH Tau b', 30.2643),
 ('Kepler-1979 b', 29.33),
 ('TOI-1408 b', 25.0),
 ('CT Cha b', 24.66),
 ('HAT-P-67 b', 23.9872187)]

### 5) Compara `COUNT(*)` vs `COUNT(disc_year)` por método

In [16]:
# TODO (5): Compara COUNT(*) vs COUNT(disc_year) por método
# Pistas: GROUP BY discoverymethod, calcula total y non_null_year = COUNT(disc_year)

query = """
SELECT
  discoverymethod,
  COUNT(*) AS total,
  COUNT(disc_year) AS non_null_year
FROM raw_ps
GROUP BY discoverymethod
ORDER BY total DESC
"""
con.execute(query).fetchall()

[('Transit', 4501, 4500),
 ('Radial Velocity', 1166, 1166),
 ('Microlensing', 266, 266),
 ('Imaging', 92, 92),
 ('Transit Timing Variations', 39, 39),
 ('Eclipse Timing Variations', 17, 17),
 ('Orbital Brightness Modulation', 9, 9),
 ('Pulsar Timing', 8, 8),
 ('Astrometry', 6, 6),
 ('Pulsation Timing Variations', 2, 2),
 ('Disk Kinematics', 1, 1)]

### 6) Resumen: por método, n_planets y mediana de periodo orbital

In [17]:
# TODO (6): Resumen por método: n_planets y mediana del periodo orbital
# Pistas: MEDIAN(pl_orbper) (filtra NULL si aplica), GROUP BY discoverymethod
query = """
SELECT
  discoverymethod,
  COUNT(*) AS n_planets,
  MEDIAN(pl_orbper) AS mediana_periodo
FROM raw_ps
WHERE pl_orbper IS NOT NULL
GROUP BY discoverymethod
ORDER BY n_planets DESC
"""
con.execute(query).fetchall()

[('Transit', 4501, 8.15872),
 ('Radial Velocity', 1166, 298.895),
 ('Transit Timing Variations', 39, 30.0),
 ('Imaging', 25, 33000.0),
 ('Eclipse Timing Variations', 17, 3160.0),
 ('Microlensing', 12, 3142.5),
 ('Orbital Brightness Modulation', 9, 0.81161),
 ('Pulsar Timing', 7, 25.262),
 ('Astrometry', 6, 334.76),
 ('Pulsation Timing Variations', 2, 1005.0)]

## Para entregar (tarea)
1) **4 consultas adicionales** (tú decides las preguntas), pero deben incluir:
   - 2 consultas de calidad (nulos, rangos, duplicados, outliers simples)
   - 2 consultas científicas: pregunta + 1–2 líneas de interpretación 

2) En `docs/decisions_log.md`: 1 decisión de hoy (con evidencia: conteos o query).

## Reflexión (bitácora)
- ¿Qué consulta te pareció más difícil y por qué?
- Si el dataset creciera 100×, ¿qué consultas crees que empeoran más?


**Entrega sugerida:** crea `docs/w02a_sql_practice.md` y pega tus 6 respuestas (1–6) + 4 consultas adicionales tuyas con resultados.


Calidad 1 — Duplicados por nombre de planeta

In [18]:
query = """
SELECT pl_name, COUNT(*) AS veces
FROM raw_ps
GROUP BY pl_name
HAVING COUNT(*) > 1
ORDER BY veces DESC
"""
con.execute(query).fetchall()

[]

Calidad 2 — Outliers de radio (planetas enormes, mayores a 30 radios terrestres)

In [19]:
query = """
SELECT pl_name, pl_rade
FROM raw_ps
WHERE pl_rade > 30
ORDER BY pl_rade DESC
"""
con.execute(query).fetchall()

[('V2376 Ori b', 87.20586985),
 ('HD 100546 b', 77.3421),
 ('GQ Lup b', 33.6),
 ('Kepler-297 d', 32.6),
 ('PDS 70 b', 30.48848),
 ('DH Tau b', 30.2643)]

Cientifica 1 — Temperatura promedio por método de descubrimiento

In [20]:
query = """
SELECT
  discoverymethod,
  ROUND(AVG(pl_eqt), 2) AS temp_promedio_k
FROM raw_ps
WHERE pl_eqt IS NOT NULL
GROUP BY discoverymethod
ORDER BY temp_promedio_k DESC
"""
con.execute(query).fetchall()

[('Orbital Brightness Modulation', 2140.0),
 ('Imaging', 1577.27),
 ('Transit', 922.9),
 ('Transit Timing Variations', 684.43),
 ('Radial Velocity', 558.46),
 ('Microlensing', 56.0)]

Científica 2 — Sistemas con más de 3 planetas y su distancia promedio

In [21]:
query = """
SELECT
  hostname,
  COUNT(*) AS n_planetas,
  ROUND(AVG(sy_dist), 2) AS dist_promedio_pc
FROM raw_ps
GROUP BY hostname
HAVING COUNT(*) > 3
ORDER BY n_planetas DESC
"""
con.execute(query).fetchall()

[('KOI-351', 8, 848.25),
 ('TRAPPIST-1', 7, 12.43),
 ('HD 34445', 6, 46.09),
 ('Kepler-80', 6, 369.45),
 ('HD 110067', 6, 32.16),
 ('TOI-178', 6, 62.7),
 ('TOI-1136', 6, 84.54),
 ('K2-138', 6, 202.59),
 ('HD 10180', 6, 38.96),
 ('Kepler-20', 6, 282.56),
 ('HD 191939', 6, 53.61),
 ('HD 219134', 6, 6.53),
 ('Kepler-11', 6, 646.35),
 ('HIP 41378', 6, 106.29),
 ('Kepler-33', 5, 1209.16),
 ('K2-384', 5, 82.66),
 ('Kepler-238', 5, 1798.75),
 ('GJ 667 C', 5, 7.24),
 ('Kepler-82', 5, 904.33),
 ('55 Cnc', 5, 12.59),
 ('HD 158259', 5, 27.05),
 ('HD 40307', 5, 12.94),
 ('Kepler-122', 5, 1027.45),
 ('Kepler-32', 5, 323.85),
 ('Kepler-186', 5, 177.59),
 ('Kepler-139', 5, 391.04),
 ('Kepler-62', 5, 300.87),
 ('Kepler-292', 5, 1056.48),
 ('HD 134606', 5, 26.79),
 ('Kepler-150', 5, 891.09),
 ('Kepler-154', 5, 915.21),
 ('HD 108236', 5, 64.6),
 ('Kepler-296', 5, 167.0),
 ('L 98-59', 5, 10.62),
 ('Kepler-169', 5, 406.53),
 ('HD 23472', 5, 39.03),
 ('Kepler-48', 5, 306.74),
 ('Kepler-102', 5, 107.8),
 ('